In [ ]:
import base64
import os
# Install the required package
# %pip install google-genai
from google import genai
from google.genai import types
import google.generativeai as genai
from pydantic import BaseModel
from dotenv import load_dotenv
import tqdm as notebook_tqdm
import json
import pprint as pp
import dotenv as env
import sys
import pandas

# Load environment variables
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
print(api_key)
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment variables.")

# Configure Gemini API
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# List available models for debugging
print("Available models:")
for model in genai.list_models():
    print(f"- {model.name}")

In [ ]:
class HeadlineClassification(BaseModel):
    link_id: str
    title: str                          #Do not change the name of this field as OpenAI is dumb. If you keep it as inputs, then only it returns the original text sent to it.
                                        # Will have to carefully write wrappers to ensure that the outcome df to be used in the final output has the right headers. 
    classification: bool
    explanation: str

In [ ]:
import os
import sys
import json
import pandas as pd
import google.generativeai as genai
from typing import TypedDict, List

class HeadlineClassification(TypedDict):
    category: str
    confidence: float
    reason: str

class Gemini_Models:
    # Use the correct model name as discovered in Kernel 1
    __model_name = 'gemini-2.0-flash'  # Updated to match the model you have access to
    __prompt_file_path = r"C:\Users\vanshika.alang\Desktop\re_news_feed-main-updated\prompts\gemini_ai_prompts.json"
    
    def __init__(self): # Fixed initialization method syntax
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("Missing GEMINI_API_KEY environment variable.")
        
        genai.configure(api_key=api_key)
        self.__client = genai.GenerativeModel(self.__model_name)
    
    def __getPromptFromFile(self, type: str) -> str:
        with open(self.__prompt_file_path, 'r') as file:
            parsed_data = json.load(file)
        prompt = parsed_data.get(type)
        if not prompt:
            raise ValueError(f"No prompt found for type '{type}' in {self.__prompt_file_path}")
        return prompt
    
    def classify_headlines(self, input: pd.DataFrame, silent_mode=True) -> str:
        if silent_mode:
            original_stdout = sys.stdout
            sys.stdout = open(os.devnull, 'w')
        
        try:
            prompt = self.__getPromptFromFile('headlines_classifier_real_estate')
            print('Sending titles for classification to Gemini model...')
            print(f'Input is of length: {len(input)}.')
            
            # Create the generation config
            generation_config = {
                "temperature": 0.8,
                "response_mime_type": "application/json"
            }
            
            # Get the data to send
            data_to_send = input.to_json(orient='records')
            
            # Create content with system prompt and user data
            response = self.__client.generate_content(
                contents=[
                    {"role": "system", "parts": [prompt]},
                    {"role": "user", "parts": [data_to_send]}
                ],
                generation_config=generation_config
            )
            
            return response.text
        except Exception as e:
            print(f'Gemini AI execution threw an exception: {e}')
            return None
        finally:
            if silent_mode:
                sys.stdout.close()
                sys.stdout = original_stdout

In [1]:
import os
import sys
import json
import pandas as pd
import google.generativeai as genai
from typing import TypedDict, List
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

class HeadlineClassification(TypedDict):
    category: str
    confidence: float
    reason: str

class Gemini_Models:
    # Use the correct model name
    __model_name = 'gemini-2.0-flash'
    __prompt_file_path = r"C:\Users\vanshika.alang\Desktop\re_news_feed-main-updated\prompts\gemini_ai_prompts.json"
    
    def __init__(self):
        # Try to get API key from environment variable
        api_key = os.environ.get("GEMINI_API_KEY") or os.getenv("GEMINI_API_KEY")
        
        if not api_key:
            print("Warning: No API key found. Please enter your API key below:")
            api_key = input("Enter your Gemini API key: ")
            if not api_key:
                raise ValueError("No API key provided")
        
        genai.configure(api_key=api_key)
        self.__client = genai.GenerativeModel(self.__model_name)
    
    def __getPromptFromFile(self, type: str) -> str:
        try:
            with open(self.__prompt_file_path, 'r') as file:
                parsed_data = json.load(file)
            prompt = parsed_data.get(type)
            if not prompt:
                raise ValueError(f"No prompt found for type '{type}' in {self.__prompt_file_path}")
            return prompt
        except FileNotFoundError:
            # Provide a default prompt if file not found
            print(f"Warning: Prompt file not found at {self.__prompt_file_path}")
            return "You are an AI assistant that classifies news headlines. Please classify the following headlines as related to real estate (true) or not related to real estate (false). Provide your answer as a JSON array with 'title', 'classification', and 'explanation' fields."
    
    def classify_headlines(self, input: pd.DataFrame, silent_mode=True) -> str:
        if silent_mode:
            original_stdout = sys.stdout
            sys.stdout = open(os.devnull, 'w')
        
        try:
            prompt = self.__getPromptFromFile('headlines_classifier_real_estate')
            print('Sending titles for classification to Gemini model...')
            print(f'Input is of length: {len(input)}.')
            
            # Create the generation config
            generation_config = {
                "temperature": 0.8,
                "response_mime_type": "application/json"
            }
            
            # Get the data to send
            data_to_send = input.to_json(orient='records')
            
            # Combine prompt and data without using system role (not supported in this model)
            combined_prompt = f"{prompt}\n\nHere are the headlines to classify:\n{data_to_send}"
            
            # Call the API with a single content part (not using system role)
            response = self.__client.generate_content(
                combined_prompt,
                generation_config=generation_config
            )
            
            return response.text
        except Exception as e:
            print(f'Gemini AI execution threw an exception: {e}')
            return None
        finally:
            if silent_mode:
                sys.stdout.close()
                sys.stdout = original_stdout

# Now test the implementation
llm = Gemini_Models()
df = pd.DataFrame([
    ['Mumbai', 'Ghatkopar college student among two to drown in sea'],
    ['Mumbai', 'NCP-SP MLA Jitendra Awhad receives death threats'],
    ['Mumbai', 'Mumbai court says life term for rapist dad too harsh']
], columns=['sub-site', 'title'])

response = llm.classify_headlines(df[['sub-site', 'title']], silent_mode=False)

if response:
    try:
        out_df = pd.DataFrame(json.loads(response))
        print(out_df)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON response: {e}")
        print(f"Raw response: {response}")
else:
    print("No response received from the model")

C:\Users\vanshika.alang\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sending titles for classification to Gemini model...
Input is of length: 3.
  Link_ID                                           Headline  Classification  \
0    None  Ghatkopar college student among two to drown i...           False   
1    None   NCP-SP MLA Jitendra Awhad receives death threats           False   
2    None  Mumbai court says life term for rapist dad too...           False   

                                         Explanation  
0  The headline discusses a drowning incident and...  
1  The headline is about death threats to a polit...  
2  The headline discusses a court case, unrelated...  
